# Демо · Вопрос по материалам курса

Лекция 5, слайд демо. Тезис: модель во всех шагах одна и та же, а ответ системы определяется тем,
что код положил в контекст до вызова и что проверил после. Чат-версия `Qwen3-0.6B`, режим размышлений выключен,
для поиска энкодер `multilingual-e5-small`. Корпус это материалы курса: заметки к лекциям 1–4, задания
лабораторных 1–4, ТЗ курсовой, регламент.

Пять шагов:

1. Вопрос, ответ на который есть только в материалах курса. Модель без поиска.
2. Поиск: пять ближайших фрагментов.
3. Тот же вопрос с фрагментами в промпте.
4. Вопрос, ответа на который в материалах нет. Поиск всё равно что-то находит.
5. Проверка двух ответов: сначала спрашиваем модель, потом решает код.

Все выводы сохраняются в `fallback/`.

In [1]:
import json
import os
import re
import urllib.request
from pathlib import Path

import numpy
import torch
from transformers import AutoModel
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

CHAT_MODEL = os.environ.get("LAB_CHAT_MODEL", "Qwen/Qwen3-0.6B")
ENCODER = os.environ.get("LAB_ENCODER", "intfloat/multilingual-e5-small")
CORPUS_URL = "https://raw.githubusercontent.com/pmenshih/tgo-course/main/assignments/05/data/corpus.json"
CORPUS_PATH = Path.cwd().parents[1] / "assignments" / "05" / "data" / "corpus.json"

FALLBACK_PATH = Path.cwd() / "fallback"
NO_SEARCH_PATH = FALLBACK_PATH / "01_no_search.md"
SEARCH_PATH = FALLBACK_PATH / "02_search.md"
WITH_FRAGMENTS_PATH = FALLBACK_PATH / "03_with_fragments.md"
NO_ANSWER_PATH = FALLBACK_PATH / "04_no_answer.md"
CHECK_PATH = FALLBACK_PATH / "05_check.md"
FALLBACK_PATH.mkdir(exist_ok=True)

TOP_K = 5
CHUNK_WORD_LIMIT = 120
PREVIEW_LENGTH = 90
REFUSAL = "нет данных"
JSON_PATTERN = re.compile(r"\{.*?\}", re.DOTALL)

COURSE = "«Технологии глубокого обучения и нейронных сетей»"
NO_SEARCH_SYSTEM = (
    f"Ты отвечаешь на вопросы о курсе {COURSE}. "
    'Ответь коротко, одной строкой JSON вида {"answer": "...", "source": null}. '
    'Если не знаешь ответа, напиши {"answer": "нет данных", "source": null}.'
)
RAG_SYSTEM = (
    f"Ты отвечаешь на вопросы о курсе {COURSE} только по фрагментам из сообщения пользователя. "
    'Ответь коротко, одной строкой JSON вида {"answer": "...", "source": номер фрагмента}. '
    'Если во фрагментах нет ответа, напиши {"answer": "нет данных", "source": null}.'
)
JUDGE_SYSTEM = (
    "Ты проверяешь ответ на вопрос по фрагментам документов. "
    "Ответь одним словом: «да», если ответ прямо следует из фрагментов, и «нет», если не следует."
)

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
DTYPE = torch.float32 if DEVICE == "cpu" else torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, dtype=DTYPE).to(DEVICE).eval()
encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER)
encoder = AutoModel.from_pretrained(ENCODER).to(DEVICE).eval()
print("устройство", DEVICE, "| модель", CHAT_MODEL, "| энкодер", ENCODER)


def system(text: str) -> dict:
    return {"role": "system", "content": text}


def user(text: str) -> dict:
    return {"role": "user", "content": text}


def load_documents() -> list[dict]:
    """Корпус из репозитория курса; если файла рядом нет, он скачивается."""
    if CORPUS_PATH.exists():
        text = CORPUS_PATH.read_text(encoding="utf-8")
    else:
        with urllib.request.urlopen(CORPUS_URL) as response:
            text = response.read().decode("utf-8")
    return json.loads(text)["documents"]


def split_document(document: dict) -> list[dict]:
    """Блоки по пустым строкам склеиваются под одним заголовком, пока в фрагменте не больше 120 слов."""
    chunks = []
    heading = ""
    blocks = []

    def close_chunk() -> None:
        if blocks:
            chunk = {
                "doc": document["doc"],
                "heading": heading,
                "text": "\n\n".join(blocks),
            }
            chunks.append(chunk)
            blocks.clear()

    for block in document["text"].split("\n\n"):
        block = block.strip()
        if not block:
            continue

        if block.startswith("#"):
            close_chunk()
            first_line, _, rest = block.partition("\n")
            heading = first_line.lstrip("#").strip()
            block = rest.strip()
            if not block:
                continue

        words_so_far = sum(len(part.split()) for part in blocks)
        if blocks and words_so_far + len(block.split()) > CHUNK_WORD_LIMIT:
            close_chunk()
        blocks.append(block)

    close_chunk()
    return chunks


def passage(chunk: dict) -> str:
    if chunk["heading"]:
        return chunk["heading"] + "\n" + chunk["text"]
    return chunk["text"]


@torch.no_grad()
def embed(texts: list[str], prefix: str) -> numpy.ndarray:
    """Векторы e5: префикс, среднее выходов по маске, единичная длина."""
    vectors = []
    for start in range(0, len(texts), 16):
        batch = [prefix + text for text in texts[start:start + 16]]
        inputs = encoder_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        hidden = encoder(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        mean = torch.nn.functional.normalize(mean, dim=-1)
        vectors.append(mean.float().cpu().numpy())
    return numpy.concatenate(vectors)


def search(query: str, k: int = TOP_K) -> list[tuple[int, float]]:
    """Номера k ближайших фрагментов и косинусы."""
    query_vector = embed([query], "query: ")[0]
    scores = chunk_vectors @ query_vector
    top = numpy.argsort(-scores)[:k]
    return [(int(chunk_id), float(scores[chunk_id])) for chunk_id in top]


def format_fragments(chunk_ids: list[int]) -> str:
    lines = []
    for number, chunk_id in enumerate(chunk_ids, start=1):
        lines.append(f"[{number}] {passage(chunks[chunk_id])}")
    return "Фрагменты:\n" + "\n\n".join(lines)


def describe_fragments(found: list[tuple[int, float]]) -> str:
    """Таблица найденного: номер, косинус, документ, заголовок, начало текста."""
    lines = []
    for number, (chunk_id, score) in enumerate(found, start=1):
        chunk = chunks[chunk_id]
        preview = " ".join(chunk["text"].split())[:PREVIEW_LENGTH]
        lines.append(f"[{number}] {score:.3f} | {chunk['doc']} | {chunk['heading']}\n    {preview}…")
    return "\n".join(lines)


@torch.no_grad()
def chat(messages: list[dict], max_new_tokens: int = 64) -> tuple[str, int]:
    """Жадный ответ модели и длина промпта в токенах."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_length = inputs["input_ids"].shape[1]
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True).strip()
    return answer, prompt_length


def parse_answer(raw: str) -> tuple[str, int | None]:
    """Поля answer и source из JSON-строки ответа; пустой ответ, если разобрать не удалось."""
    match = JSON_PATTERN.search(raw)
    if match is None:
        return "", None
    try:
        parsed = json.loads(match.group())
    except json.JSONDecodeError:
        return "", None
    else:
        source = parsed.get("source")
        if not isinstance(source, int):
            source = None
        return str(parsed.get("answer", "")).strip(), source


def normalize(text: str) -> str:
    lowered = text.lower().replace("ё", "е")
    return "".join(symbol for symbol in lowered if symbol.isalnum())


def ask_with_fragments(question: str, chunk_ids: list[int]) -> tuple[str, int]:
    messages = [
        system(RAG_SYSTEM),
        user(f"{format_fragments(chunk_ids)}\n\nВопрос: {question}"),
    ]
    return chat(messages)


def ask_judge(question: str, chunk_ids: list[int], answer: str) -> str:
    """Спросить саму модель, следует ли ответ из фрагментов."""
    messages = [
        system(JUDGE_SYSTEM),
        user(
            f"{format_fragments(chunk_ids)}\n\nВопрос: {question}\nОтвет: {answer}\n\n"
            "Следует ли этот ответ из фрагментов? Ответь «да» или «нет»."
        ),
    ]
    verdict, _ = chat(messages, max_new_tokens=4)
    return verdict


def check_by_repeat(question: str, chunk_ids: list[int], raw: str) -> dict:
    """Повторный вопрос по одному фрагменту, на который сослалась модель; решает код."""
    answer, source = parse_answer(raw)
    if source is None or not 1 <= source <= len(chunk_ids):
        return {"second": "", "passed": False, "reason": "нет номера фрагмента"}

    second_raw, _ = ask_with_fragments(question, [chunk_ids[source - 1]])
    second, _ = parse_answer(second_raw)
    first_normalized = normalize(answer)
    second_normalized = normalize(second)
    same = bool(second_normalized) and (
        first_normalized in second_normalized or second_normalized in first_normalized
    )
    if normalize(REFUSAL) in second_normalized:
        return {"second": second_raw, "passed": False, "reason": "при повторе модель отказалась"}
    if not same:
        return {"second": second_raw, "passed": False, "reason": "при повторе другой ответ"}
    return {"second": second_raw, "passed": True, "reason": "ответы совпали"}


documents = load_documents()
chunks = []
for document in documents:
    chunks.extend(split_document(document))
chunk_vectors = embed([passage(chunk) for chunk in chunks], "passage: ")
corpus_tokens = sum(len(tokenizer(document["text"])["input_ids"]) for document in documents)
print(f"документов {len(documents)}, фрагментов {len(chunks)}, токенов в корпусе {corpus_tokens}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

устройство mps | модель Qwen/Qwen3-0.6B | энкодер intfloat/multilingual-e5-small
документов 11, фрагментов 207, токенов в корпусе 44126


## 1. Вопрос без поиска

Ответ на этот вопрос есть в заметке ко второй лекции. В весах модели его нет: модель видит только инструкцию
и вопрос.

In [2]:
QUESTION = os.environ.get("DEMO_QUESTION", "Каков размер словаря модели Qwen3-0.6B-Base в токенах?")

messages = [
    system(NO_SEARCH_SYSTEM),
    user(f"Вопрос: {QUESTION}"),
]
answer_without_search, tokens_without_search = chat(messages)
print("Вопрос:", QUESTION)
print("Модель без поиска:", answer_without_search)
print("Промпт, токенов:", tokens_without_search)

NO_SEARCH_PATH.write_text(
    f"**Вопрос:** {QUESTION}\n\n"
    f"**Модель без поиска:** `{answer_without_search}`\n\n"
    f"Промпт: {tokens_without_search} токенов\n",
    encoding="utf-8",
)

Вопрос: Каков размер словаря модели Qwen3-0.6B-Base в токенах?
Модель без поиска: {"answer": "1024", "source": null}
Промпт, токенов: 119


149

## 2. Поиск

Вопрос превращается в вектор, и код находит пять фрагментов с самым большим косинусом. Языковая модель
в этом шаге не участвует.

In [3]:
found = search(QUESTION)
found_ids = [chunk_id for chunk_id, _ in found]
found_description = describe_fragments(found)
print(found_description)

SEARCH_PATH.write_text(
    f"**Вопрос:** {QUESTION}\n\n```\n{found_description}\n```\n",
    encoding="utf-8",
)

[1] 0.897 | lectures/02/why-attention.md | Внимание в матрицах: что стоит за одной строкой
    Как читать. Числа в примерах взяты для `Qwen3-0.6B-Base`, той же модели, что в демо и в ла…
[2] 0.891 | assignments/02/README.md | Что сделать
    2. **Токенизатор настоящей модели и бюджет контекста.** Взять токенизатор Qwen3 через `tra…
[3] 0.880 | assignments/03/README.md | Бонус
    - **Три точки на одном тексте.** Взять первые 2000 слов отложенной части и посчитать, скол…
[4] 0.871 | lectures/02/why-attention.md | 1. Вход: матрица токенов
    Текст из $n$ токенов после таблицы эмбеддингов это $n$ векторов размерности $d$. Записывае…
[5] 0.870 | assignments/02/README.md | Бонус
    - **Испорченный телефон.** Та же инструкция и тот же наполнитель, что в части 4б, только т…


856

## 3. Тот же вопрос с фрагментами

Код кладет пять фрагментов в промпт перед вопросом и просит сослаться на номер фрагмента. Модель та же.

In [4]:
answer_with_fragments, tokens_with_fragments = ask_with_fragments(QUESTION, found_ids)
print("Модель с фрагментами:", answer_with_fragments)
print("Промпт, токенов:", tokens_with_fragments, "против", tokens_without_search, "без поиска")

WITH_FRAGMENTS_PATH.write_text(
    f"**Вопрос:** {QUESTION}\n\n"
    f"**Модель без поиска:** `{answer_without_search}`\n\n"
    f"**Модель с фрагментами:** `{answer_with_fragments}`\n\n"
    f"Промпт: {tokens_with_fragments} токенов против {tokens_without_search} без поиска\n",
    encoding="utf-8",
)

Модель с фрагментами: {"answer": "151 936", "source": 1}
Промпт, токенов: 1750 против 119 без поиска


236

## 4. Вопрос, на который в материалах нет ответа

Про эту модель в материалах курса не сказано, сколько у нее слоев. Поиск об этом не знает: он всегда
возвращает пять ближайших фрагментов. В инструкции написано, что делать, если ответа нет.

In [5]:
UNANSWERABLE = os.environ.get("DEMO_UNANSWERABLE", "Сколько слоев у модели multilingual-e5-small?")

found_unanswerable = search(UNANSWERABLE)
unanswerable_ids = [chunk_id for chunk_id, _ in found_unanswerable]
unanswerable_description = describe_fragments(found_unanswerable)
answer_unanswerable, _ = ask_with_fragments(UNANSWERABLE, unanswerable_ids)
print("Вопрос:", UNANSWERABLE)
print(unanswerable_description)
print("\nМодель с фрагментами:", answer_unanswerable)

NO_ANSWER_PATH.write_text(
    f"**Вопрос:** {UNANSWERABLE}\n\n"
    f"```\n{unanswerable_description}\n```\n\n"
    f"**Модель с фрагментами:** `{answer_unanswerable}`\n",
    encoding="utf-8",
)

Вопрос: Сколько слоев у модели multilingual-e5-small?
[1] 0.848 | lectures/03/how-training-works.md | 5. Законы масштабирования
    Третий сдвиг уже после Chinchilla: оптимум по обучению не равен оптимуму по эксплуатации. …
[2] 0.847 | lectures/02/why-attention.md | 8. Слой целиком: остаточные связи и нормализация
    где $W_1, W_3$ размера $d \times d_{ff}$, $W_2$ размера $d_{ff} \times d$, $\sigma$ это Si…
[3] 0.839 | Курсовая.md | 2.4. Ограничения на модель
    - Основная языковая модель: не более 4 миллиардов параметров. Ориентиры: Qwen3-4B, Qwen3-1…
[4] 0.839 | lectures/02/why-attention.md | Внимание в матрицах: что стоит за одной строкой
    Как читать. Числа в примерах взяты для `Qwen3-0.6B-Base`, той же модели, что в демо и в ла…
[5] 0.836 | lectures/02/why-attention.md | 11. Где лежат 600 миллионов
    | Что | Размер | Параметров | |---|---|---| | Таблица эмбеддингов | $151\,936 \times 1024$…

Модель с фрагментами: {"answer": "28", "source": 4}.


988

## 5. Проверка

Два ответа, один верный, другой выдуман. Сначала спросим саму модель, следует ли ответ из фрагментов.
Потом зададим тот же вопрос второй раз, оставив в промпте один фрагмент, на который модель сослалась,
и сравним два ответа кодом.

In [6]:
cases = [
    ("вопрос с ответом", QUESTION, found_ids, answer_with_fragments),
    ("вопрос без ответа", UNANSWERABLE, unanswerable_ids, answer_unanswerable),
]

lines = []
for label, question, chunk_ids, raw in cases:
    answer, _ = parse_answer(raw)
    judge_verdict = ask_judge(question, chunk_ids, answer)
    check = check_by_repeat(question, chunk_ids, raw)
    final_answer = answer if check["passed"] else REFUSAL

    case_lines = [
        f"{label}: {question}",
        f"  ответ модели:        {raw}",
        f"  модель о себе:       {judge_verdict}",
        f"  повторный ответ:     {check['second']}",
        f"  решение кода:        {check['reason']}",
        f"  ответ пользователю:  {final_answer}",
    ]
    print("\n".join(case_lines))
    print()
    lines.extend(case_lines)
    lines.append("")

CHECK_PATH.write_text("```\n" + "\n".join(lines) + "```\n", encoding="utf-8")

вопрос с ответом: Каков размер словаря модели Qwen3-0.6B-Base в токенах?
  ответ модели:        {"answer": "151 936", "source": 1}
  модель о себе:       да
  повторный ответ:     {"answer": "151 936", "source": 1}
  решение кода:        ответы совпали
  ответ пользователю:  151 936

вопрос без ответа: Сколько слоев у модели multilingual-e5-small?
  ответ модели:        {"answer": "28", "source": 4}.
  модель о себе:       да
  повторный ответ:     {"answer": "16", "source": 1}
  решение кода:        при повторе другой ответ
  ответ пользователю:  нет данных



573